## ONNX 환경
```
onnxruntime-gpu           1.16.3  (GPU 버전)
NumPy 1.26.4 + ONNX 1.16.0 + ORT 1.16.3 + onnxscript 0.1.0.dev20231031
```

- CUDA: v11.8 설치 완료 및 환경 변수 등록
- cuDNN: v11.8 폴더 안에 DLL 파일 복사 완료
- NumPy: 1.26.4 버전 ( 1.x 대여야 함)


### Python에서 JSON을 다루는 방법

In [2]:
import json

# Python dict → JSON 문자열 (직렬화)
data = {
    "text": "이 영화 정말 재밌다",
    "return_probabilities": True,
    "max_length": None
}

json_string = json.dumps(data, ensure_ascii=False, indent=2)
print("=== Python → JSON ===")
print(json_string)
print(f"타입: {type(json_string)}")  

=== Python → JSON ===
{
  "text": "이 영화 정말 재밌다",
  "return_probabilities": true,
  "max_length": null
}
타입: <class 'str'>


In [4]:
# JSON 문자열 → Python dict (역직렬화)
parsed = json.loads(json_string)
print("\n=== JSON → Python ===")
print(parsed)
print(f"타입: {type(parsed)}")       #
print(f"텍스트: {parsed['text']}")   # 이 영화 정말 재밌다


=== JSON → Python ===
{'text': '이 영화 정말 재밌다', 'return_probabilities': True, 'max_length': None}
타입: <class 'dict'>
텍스트: 이 영화 정말 재밌다


### 실습 : 실제 API 호출

In [5]:
import requests

# JSONPlaceholder: 테스트용 공개 REST API
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

print(f"상태 코드: {response.status_code}")   # 200
print(f"응답 타입: {type(response.json())}")   #
print(f"응답 내용:")
print(json.dumps(response.json(), indent=2))


상태 코드: 200
응답 타입: <class 'dict'>
응답 내용:
{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


### POST 요청 — 데이터 전송

In [6]:
# POST 요청: 새로운 데이터를 전송합니다
response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json={                                # json= 을 사용하면 자동으로 JSON 변환 + 헤더 설정
        "title": "모델 배포 테스트",
        "body": "FastAPI로 모델을 서빙합니다",
        "userId": 1
    }
)

print(f"상태 코드: {response.status_code}")   # 201 (Created)
print(f"응답 내용:")
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

상태 코드: 201
응답 내용:
{
  "title": "모델 배포 테스트",
  "body": "FastAPI로 모델을 서빙합니다",
  "userId": 1,
  "id": 101
}


### 에러 상황 체험

In [7]:
# 존재하지 않는 리소스에 GET 요청
response = requests.get("https://jsonplaceholder.typicode.com/posts/99999")
print(f"상태 코드: {response.status_code}")   # 404 (Not Found)
print(f"응답 내용: {response.json()}")         # {}

상태 코드: 404
응답 내용: {}


In [8]:
# 잘못된 URL로 요청
try:
    response = requests.get("https://jsonplaceholder.typicode.com/없는경로")
    print(f"상태 코드: {response.status_code}")   # 404
except requests.exceptions.RequestException as e:
    print(f"요청 실패: {e}")

상태 코드: 404


### 직렬화 실습

In [1]:
import torch
import torch.nn as nn

In [2]:
class SimpleClassifier(nn.Module):
    """
    간단한 이미지 분류 모델
    - 입력: 1x28x28 (MNIST와 동일한 크기)
    - 출력: 10개 클래스에 대한 확률
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [3]:
# 모델 인스턴스 생성
model = SimpleClassifier(num_classes=10)

# 더미 입력으로 동작 확인
dummy_input = torch.randn(1, 1, 28, 28)  # (batch=1, channels=1, height=28, width=28)
output = model(dummy_input)

print(f"모델 구조:\n{model}\n")
print(f"입력 크기: {dummy_input.shape}")
print(f"출력 크기: {output.shape}")          # torch.Size([1, 10])
print(f"출력 값:   {output.detach()}")

모델 구조:
SimpleClassifier(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)

입력 크기: torch.Size([1, 1, 28, 28])
출력 크기: torch.Size([1, 10])
출력 값:   tensor([[-0.1167, -0.2139, -0.3197, -0.1827, -0.3083, -0.0286, -0.1337,  0.0179,
         -0.0995, -0.0569]])


In [5]:
import os

# 모델 저장 폴더 확인
os.makedirs("models", exist_ok=True)

#### 방법 1: state_dict 저장 (Pickle 기반)

In [14]:
# state_dict: 모델의 가중치(파라미터)만 딕셔너리 형태로 저장합니다

# 1. 원본 모델을 평가 모드로 전환 (매우 중요!)
model.eval()

# 2. 원본 모델에서 출력값 생성
with torch.no_grad():
    output = model(dummy_input)

# 3. 모델 저장
torch.save(model.state_dict(), "models/model_state_dict.pth")

# 파일 크기 확인
file_size = os.path.getsize("models/model_state_dict.pth")
print(f"저장 완료: models/model_state_dict.pth")
print(f"파일 크기: {file_size / 1024:.1f} KB")

저장 완료: models/model_state_dict.pth
파일 크기: 1650.5 KB


In [15]:
# 저장된 내용 확인: 어떤 키들이 들어 있는지 살펴봅니다
state_dict = torch.load("models/model_state_dict.pth", weights_only=True)

print("저장된 키 목록:")
for key, tensor in state_dict.items():
    print(f"  {key:40s} → {tensor.shape}")


저장된 키 목록:
  features.0.weight                        → torch.Size([32, 1, 3, 3])
  features.0.bias                          → torch.Size([32])
  features.3.weight                        → torch.Size([64, 32, 3, 3])
  features.3.bias                          → torch.Size([64])
  classifier.1.weight                      → torch.Size([128, 3136])
  classifier.1.bias                        → torch.Size([128])
  classifier.4.weight                      → torch.Size([10, 128])
  classifier.4.bias                        → torch.Size([10])


##### 불러오기

In [16]:
# 불러올 때는 반드시 동일한 모델 클래스가 필요합니다
# 1. 원본 모델을 평가 모드로 전환 (매우 중요!)
model.eval()

# 2. 원본 모델에서 출력값 생성
with torch.no_grad():
    output = model(dummy_input)

# 3. 모델 저장
torch.save(model.state_dict(), "models/model_state_dict.pth")

# --- 여기서부터 불러오기 ---

# 4. 새 모델 선언 및 가중치 로드
loaded_model = SimpleClassifier(num_classes=10)
loaded_model.load_state_dict(torch.load("models/model_state_dict.pth", weights_only=True))

# 5. 불러온 모델도 평가 모드로 전환
loaded_model.eval()

# 6. 복원된 모델에서 출력값 생성
with torch.no_grad():
    loaded_output = loaded_model(dummy_input)

In [17]:
# 동일한 입력에 대해 동일한 출력이 나오는지 확인
with torch.no_grad():
    loaded_output = loaded_model(dummy_input)

print(f"원본 출력:  {output.detach()}")
print(f"복원 출력:  {loaded_output}")
print(f"동일 여부:  {torch.allclose(output.detach(), loaded_output)}")  # True

원본 출력:  tensor([[-0.1014,  0.0736,  0.0126, -0.0160, -0.1380,  0.0102,  0.0350, -0.0583,
         -0.0477, -0.2989]])
복원 출력:  tensor([[-0.1014,  0.0736,  0.0126, -0.0160, -0.1380,  0.0102,  0.0350, -0.0583,
         -0.0477, -0.2989]])
동일 여부:  True


### 방법 2: TorchScript
- TorchScript는 PyTorch 모델을 Python 없이도 실행할 수 있는 형태로 변환하는 방법입니다.
```
state_dict 방식:
  .pth 파일 + 모델 클래스 정의 + Python + PyTorch  →  추론 가능

TorchScript 방식:
  .pt 파일만 있으면                  + PyTorch(또는 libtorch)  →  추론 가능
  (모델 구조가 파일 안에 포함됨)       (Python 없이도 가능)
```

In [18]:
# eval 모드로 전환 (Dropout, BatchNorm 등의 동작이 달라지므로 필수)
model.eval()

# trace: 더미 입력을 한 번 통과시켜서 모델의 연산 그래프를 기록합니다
traced_model = torch.jit.trace(model, dummy_input)

# 저장
traced_model.save("models/model_traced.pt")

file_size = os.path.getsize("models/model_traced.pt")
print(f"저장 완료: models/model_traced.pt")
print(f"파일 크기: {file_size / 1024:.1f} KB")


저장 완료: models/model_traced.pt
파일 크기: 1673.5 KB


##### 불러오기

In [19]:
# 핵심: 모델 클래스 정의가 필요 없습니다!
loaded_traced = torch.jit.load("models/model_traced.pt")

with torch.no_grad():
    traced_output = loaded_traced(dummy_input)

print(f"원본 출력:       {output.detach()}")
print(f"TorchScript 출력: {traced_output}")
print(f"동일 여부:        {torch.allclose(output.detach(), traced_output)}")  # True

원본 출력:       tensor([[-0.1014,  0.0736,  0.0126, -0.0160, -0.1380,  0.0102,  0.0350, -0.0583,
         -0.0477, -0.2989]])
TorchScript 출력: tensor([[-0.1014,  0.0736,  0.0126, -0.0160, -0.1380,  0.0102,  0.0350, -0.0583,
         -0.0477, -0.2989]])
동일 여부:        True


##### trace vs script
TorchScript에는 두 가지 변환 방식이 있습니다.

In [20]:
# 방법 A: torch.jit.trace
# - 더미 입력을 실제로 통과시켜서 연산 경로를 기록합니다.
# - 장점: 대부분의 모델에서 잘 동작합니다.
# - 단점: 입력에 따라 분기(if/else)하는 로직은 기록되지 않습니다.
traced = torch.jit.trace(model, dummy_input)

# 방법 B: torch.jit.script
# - Python 코드를 직접 분석하여 TorchScript IR로 컴파일합니다.
# - 장점: if/else, for 루프 등 동적 로직도 변환됩니다.
# - 단점: Python 문법 중 지원되지 않는 것이 있어 에러가 발생할 수 있습니다.
scripted = torch.jit.script(model)

In [21]:
# 두 방식 모두 동일한 결과를 내는지 확인
with torch.no_grad():
    trace_out = traced(dummy_input)
    script_out = scripted(dummy_input)

print(f"trace 출력:  {trace_out}")
print(f"script 출력: {script_out}")
print(f"동일 여부:   {torch.allclose(trace_out, script_out)}")  # True

trace 출력:  tensor([[-0.1014,  0.0736,  0.0126, -0.0160, -0.1380,  0.0102,  0.0350, -0.0583,
         -0.0477, -0.2989]])
script 출력: tensor([[-0.1014,  0.0736,  0.0126, -0.0160, -0.1380,  0.0102,  0.0350, -0.0583,
         -0.0477, -0.2989]])
동일 여부:   True


#####  실무 가이드

- 모델에 if/else 분기가 없다면 → torch.jit.trace를 사용합니다 (더 안정적).
- 입력에 따라 다른 연산 경로를 타는 모델이라면 → torch.jit.script를 사용합니다.
- 이 과정에서는 torch.jit.trace를 기본으로 사용합니다.

#### 방법 3: ONNX (Open Neural Network Exchange)
- ONNX는 프레임워크 간 호환성을 목표로 하는 개방형 모델 포맷입니다.

### ONNX 변환 및 저장

In [1]:
# !pip install onnx onnxscript onnxruntime

In [1]:
# kernel die
# 라이브러리(PyTorch, ONNX, Protobuf) 간의 바이너리 충돌이나 C++ 레벨의 메모리 오류일 확률이 매우 높습니다.
# 특히 Windows 환경이나 Anaconda를 사용 중이라면 이런 현상이 잦습니다. 아래 순서대로 조치해 보세요.

In [4]:
# 1. 가장 유력한 원인: Protobuf 버전 충돌
# !pip uninstall -y onnx onnxruntime protobuf
# pip install onnx onnxruntime protobuf==3.20.3
# pip install --no-cache-dir onnx onnxruntime

### onnx 실행 에러 관련 vision 낮추어 해결

- 에러 메세지:
The kernel for 직렬화_모델배포연습.ipynb appears to have died. It will restart automatically.

```
pip install onnxruntime==1.15.0

pip uninstall numpy -y
pip install "numpy<2"

# 1. 기존의 문제 있는 torch 삭제
pip uninstall -y torch torchvision torchaudio

# 2. CPU 전용이면서 비교적 호환성이 넓은 버전으로 설치
# (최신 버전 대신 2.0.0 버전대가 구형 CPU에서 더 잘 돌아갑니다)
pip install torch==2.0.0+cpu torchvision==0.15.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu

# 확인
python -c "import torch; print('PyTorch 로드 성공:', torch.__version__)"
```


In [4]:
import numpy as np
import onnx
import onnxruntime as ort

print(f"1. NumPy 버전: {np.__version__}")       # 1.26.4 확인
print(f"2. ONNX 버전: {onnx.__version__}")      # 1.15.0 확인
print(f"3. ORT-GPU 버전: {ort.__version__}")    # 1.16.3 확인

# 최종 관문: GPU(CUDA) 인식 확인
providers = ort.get_available_providers()
print(f"4. 사용 가능한 장치: {providers}")

if 'CUDAExecutionProvider' in providers:
    print("\n✅ [성공] 모든 라이브러리 궁합이 완벽합니다. 이제 모델을 돌려보세요!")
else:
    print("\n⚠️ [주의] 라이브러리는 로드되었으나 GPU를 찾지 못합니다. cuDNN 설정을 확인하세요.")

1. NumPy 버전: 1.26.4
2. ONNX 버전: 1.15.0
3. ORT-GPU 버전: 1.16.3
4. 사용 가능한 장치: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']

✅ [성공] 모든 라이브러리 궁합이 완벽합니다. 이제 모델을 돌려보세요!


In [5]:
import torch, onnx

model.eval()

torch.onnx.export(
    model,                                    # 변환할 모델
    dummy_input,                              # 더미 입력 (모델 트레이싱에 사용)
    "models/model.onnx",                      # 저장 경로
    export_params=True,                       # 가중치를 파일에 포함
    opset_version=17,                         # ONNX 연산자 버전
    input_names=["image"],                    # 입력 텐서 이름
    output_names=["prediction"],              # 출력 텐서 이름
    dynamic_axes={                            # 가변 크기 축 지정
        "image": {0: "batch_size"},           # batch 크기를 동적으로
        "prediction": {0: "batch_size"},
    }
)

# ONNX 변환 실행 : 아래 확인후 위 다시 실행
# try:
#     print("ONNX 변환을 시작합니다...")
#     torch.onnx.export(
#         model,
#         dummy_input,
#         "models/model.onnx",
#         export_params=True,
#         opset_version=13,        # 버전 17 대신 안정적인 13 추천
#         do_constant_folding=True, # 상수 최적화 여부
#         input_names=["image"],
#         output_names=["prediction"],
#     )
#     print("성공적으로 저장되었습니다: models/model.onnx")
# except Exception as e:
#     print(f"변환 중 에러 발생: {e}")

In [7]:
import os

file_size = os.path.getsize("models/model.onnx")
print(f"저장 완료: models/model.onnx")
print(f"파일 크기: {file_size / 1024:.1f} KB")

저장 완료: models/model.onnx
파일 크기: 1649.1 KB


#### ONNX 모델 검증

In [8]:
import onnx

# 모델 구조 검증
onnx_model = onnx.load("models/model.onnx")
onnx.checker.check_model(onnx_model)
print("✅ ONNX 모델 검증 통과")

# 모델 정보 확인
print(f"\n입력:")
for inp in onnx_model.graph.input:
    print(f"  이름: {inp.name}")
    shape = [d.dim_param or d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  크기: {shape}")

print(f"\n출력:")
for out in onnx_model.graph.output:
    print(f"  이름: {out.name}")
    shape = [d.dim_param or d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  크기: {shape}")

✅ ONNX 모델 검증 통과

입력:
  이름: image
  크기: ['batch_size', 1, 28, 28]

출력:
  이름: prediction
  크기: ['batch_size', 10]


#### ONNX Runtime으로 추론

In [9]:
import onnxruntime as ort
import numpy as np

# ONNX Runtime 세션 생성
session = ort.InferenceSession("models/model.onnx")

# 입력 데이터 준비 (NumPy 배열로 변환)
input_data = dummy_input.numpy()

# 추론 실행
onnx_output = session.run(
    output_names=["prediction"],
    input_feed={"image": input_data}
)

In [10]:
print(f"PyTorch 출력:       {output.detach().numpy()}")
print(f"ONNX Runtime 출력:  {onnx_output[0]}")
print(f"동일 여부 (오차 허용): {np.allclose(output.detach().numpy(), onnx_output[0], atol=1e-5)}")

PyTorch 출력:       [[-0.11670667 -0.21393865 -0.31970096 -0.18269359 -0.30834785 -0.02855597
  -0.13372052  0.01788984 -0.09950096 -0.05686712]]
ONNX Runtime 출력:  [[-0.05503296 -0.02199238 -0.25713104 -0.1222615  -0.12575409 -0.03639618
   0.00593411 -0.04428736  0.11711745  0.04125252]]
동일 여부 (오차 허용): False


## Step 2 — 모델 학습

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [12]:
# ===== 모델 정의 (섹션 4와 동일) =====
class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [13]:
# ===== 하이퍼파라미터 =====
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 3            # 실습용이므로 3 에포크만 학습합니다

# ===== 디바이스 설정 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

사용 디바이스: cpu


In [14]:
# ===== 데이터 준비 =====
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST 평균/표준편차
])

train_dataset = datasets.MNIST(
    root="data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"학습 데이터: {len(train_dataset):,}장")
print(f"테스트 데이터: {len(test_dataset):,}장")

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data\MNIST\raw\train-images-idx3-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data\MNIST\raw\train-labels-idx1-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data\MNIST\raw\t10k-images-idx3-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%

Extracting data\MNIST\raw\t10k-labels-idx1-ubyte.gz to data\MNIST\raw

학습 데이터: 60,000장
테스트 데이터: 10,000장


In [15]:
model = SimpleClassifier(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [16]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # 200 배치마다 진행 상황 출력
        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch {epoch} [{batch_idx+1}/{len(train_loader)}] "
                  f"Loss: {running_loss/(batch_idx+1):.4f} "
                  f"Acc: {100.*correct/total:.1f}%")

    # 에포크 종료 시 요약
    train_acc = 100. * correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch}/{EPOCHS} 완료 — Loss: {avg_loss:.4f}, Acc: {train_acc:.1f}%\n")

  Epoch 1 [200/938] Loss: 0.5010 Acc: 84.1%
  Epoch 1 [400/938] Loss: 0.3328 Acc: 89.6%
  Epoch 1 [600/938] Loss: 0.2621 Acc: 91.9%
  Epoch 1 [800/938] Loss: 0.2239 Acc: 93.1%
Epoch 1/3 완료 — Loss: 0.2070, Acc: 93.7%

  Epoch 2 [200/938] Loss: 0.0880 Acc: 97.3%
  Epoch 2 [400/938] Loss: 0.0861 Acc: 97.4%
  Epoch 2 [600/938] Loss: 0.0834 Acc: 97.5%
  Epoch 2 [800/938] Loss: 0.0788 Acc: 97.6%
Epoch 2/3 완료 — Loss: 0.0773, Acc: 97.6%

  Epoch 3 [200/938] Loss: 0.0589 Acc: 98.2%
  Epoch 3 [400/938] Loss: 0.0593 Acc: 98.1%
  Epoch 3 [600/938] Loss: 0.0602 Acc: 98.1%
  Epoch 3 [800/938] Loss: 0.0592 Acc: 98.2%
Epoch 3/3 완료 — Loss: 0.0588, Acc: 98.2%



In [17]:
import os
os.makedirs("models", exist_ok=True)

# 모델을 CPU로 이동 (배포 환경에서는 GPU가 없을 수 있으므로)
model_cpu = model.cpu()
model_cpu.eval()

# 추론 비교용 테스트 입력
test_input = test_dataset[0][0].unsqueeze(0)   # 첫 번째 테스트 이미지
test_label = test_dataset[0][1]                 # 정답 레이블

print(f"테스트 입력 크기: {test_input.shape}")
print(f"정답 레이블: {test_label}")

테스트 입력 크기: torch.Size([1, 1, 28, 28])
정답 레이블: 7


In [18]:
# 저장 전 원본 모델의 추론 결과를 기록해 둡니다
with torch.no_grad():
    original_output = model_cpu(test_input)
    original_pred = original_output.argmax(dim=1).item()
    original_conf = torch.softmax(original_output, dim=1).max().item()

print(f"원본 모델 예측: {original_pred} (확신도: {original_conf:.4f})")
print(f"정답:          {test_label}")
print(f"정답 여부:      {'✅ 맞음' if original_pred == test_label else '❌ 틀림'}")

원본 모델 예측: 7 (확신도: 1.0000)
정답:          7
정답 여부:      ✅ 맞음


### 방법 1: state_dict

In [19]:
# state_dict 저장
torch.save(model_cpu.state_dict(), "models/mnist_state_dict.pth")
print(f"✅ state_dict 저장 완료: {os.path.getsize('models/mnist_state_dict.pth') / 1024:.1f} KB")

✅ state_dict 저장 완료: 1650.5 KB


### 방법 2: TorchScript

In [20]:
# TorchScript 변환 및 저장
traced_model = torch.jit.trace(model_cpu, test_input)
traced_model.save("models/mnist_traced.pt")
print(f"✅ TorchScript 저장 완료: {os.path.getsize('models/mnist_traced.pt') / 1024:.1f} KB")

✅ TorchScript 저장 완료: 1673.5 KB


### 방법 3: ONNX

In [23]:
import onnxscript

In [24]:
# ONNX 변환 및 저장
torch.onnx.export(
    model_cpu,
    test_input,
    "models/mnist_model.onnx",
    export_params=True,
    opset_version=17,
    input_names=["image"],
    output_names=["prediction"],
    dynamic_axes={
        "image": {0: "batch_size"},
        "prediction": {0: "batch_size"},
    }
)
print(f"✅ ONNX 저장 완료: {os.path.getsize('models/mnist_model.onnx') / 1024:.1f} KB")

✅ ONNX 저장 완료: 1649.1 KB


In [25]:
# 저장 결과 요약
print("\n" + "=" * 50)
print("📁 models/ 폴더 내용")
print("=" * 50)
for fname in sorted(os.listdir("models")):
    fpath = os.path.join("models", fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<30} {size_kb:>8.1f} KB")


📁 models/ 폴더 내용
  mnist_model.onnx                 1649.1 KB
  mnist_state_dict.pth             1650.5 KB
  mnist_traced.pt                  1673.5 KB
  model.onnx                       1649.1 KB


## 불러오기 및 추론 검증

### 검증 1: state_dict

In [26]:
# 클래스 정의가 반드시 있어야 합니다
loaded_sd = SimpleClassifier(num_classes=10)
loaded_sd.load_state_dict(
    torch.load("models/mnist_state_dict.pth", weights_only=True)
)
loaded_sd.eval()

with torch.no_grad():
    sd_output = loaded_sd(test_input)
    sd_pred = sd_output.argmax(dim=1).item()

print(f"[state_dict] 예측: {sd_pred}, 원본과 일치: {torch.allclose(original_output, sd_output)}")

[state_dict] 예측: 7, 원본과 일치: True


### 검증 2: TorchScript

In [27]:
# 클래스 정의가 필요 없습니다
loaded_ts = torch.jit.load("models/mnist_traced.pt")

with torch.no_grad():
    ts_output = loaded_ts(test_input)
    ts_pred = ts_output.argmax(dim=1).item()

print(f"[TorchScript] 예측: {ts_pred}, 원본과 일치: {torch.allclose(original_output, ts_output)}")

[TorchScript] 예측: 7, 원본과 일치: True


### 검증 3: ONNX

In [28]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("models/mnist_model.onnx")
onnx_output = session.run(
    ["prediction"],
    {"image": test_input.numpy()}
)

onnx_pred = np.argmax(onnx_output[0], axis=1)[0]
match = np.allclose(original_output.numpy(), onnx_output[0], atol=1e-5)

print(f"[ONNX]        예측: {onnx_pred}, 원본과 일치 (오차 허용): {match}")

[ONNX]        예측: 7, 원본과 일치 (오차 허용): True


### 종합 검증 결과

In [30]:
print("\n" + "=" * 60)
print("📊 직렬화 검증 결과 요약")
print("=" * 60)
print(f"  정답 레이블:        {test_label}")
print(f"  원본 모델 예측:     {original_pred}")
print(f"  state_dict 예측:   {sd_pred}  {'✅' if sd_pred == original_pred else '❌'}")
print(f"  TorchScript 예측:  {ts_pred}  {'✅' if ts_pred == original_pred else '❌'}")
print(f"  ONNX 예측:         {onnx_pred}  {'✅' if onnx_pred == original_pred else '❌'}")
print("=" * 60)

if all(p == original_pred for p in [sd_pred, ts_pred, onnx_pred]):
    print("\n🎉 세 가지 방식 모두 원본과 동일한 결과를 반환합니다.")
    print("   모델을 안전하게 직렬화하고 복원할 수 있다는 것이 검증되었습니다.")


📊 직렬화 검증 결과 요약
  정답 레이블:        7
  원본 모델 예측:     7
  state_dict 예측:   7  ✅
  TorchScript 예측:  7  ✅
  ONNX 예측:         7  ✅

🎉 세 가지 방식 모두 원본과 동일한 결과를 반환합니다.
   모델을 안전하게 직렬화하고 복원할 수 있다는 것이 검증되었습니다.


## 배치 추론 테스트

In [31]:
# 테스트 데이터에서 8장을 배치로 묶습니다
batch_images = torch.stack([test_dataset[i][0] for i in range(8)])
batch_labels = [test_dataset[i][1] for i in range(8)]

print(f"배치 입력 크기: {batch_images.shape}")  # torch.Size([8, 1, 28, 28])

배치 입력 크기: torch.Size([8, 1, 28, 28])


In [32]:
# 세 가지 방식으로 배치 추론
with torch.no_grad():
    sd_batch = loaded_sd(batch_images).argmax(dim=1).tolist()
    ts_batch = loaded_ts(batch_images).argmax(dim=1).tolist()

onnx_batch_out = session.run(["prediction"], {"image": batch_images.numpy()})
onnx_batch = np.argmax(onnx_batch_out[0], axis=1).tolist()

# 결과 비교
print(f"\n{'이미지':<8} {'정답':<6} {'state_dict':<12} {'TorchScript':<13} {'ONNX':<8}")
print("-" * 50)
for i in range(8):
    match = "✅" if sd_batch[i] == ts_batch[i] == onnx_batch[i] == batch_labels[i] else "❌"
    print(f"  #{i:<5} {batch_labels[i]:<6} {sd_batch[i]:<12} {ts_batch[i]:<13} {onnx_batch[i]:<8} {match}")


이미지      정답     state_dict   TorchScript   ONNX    
--------------------------------------------------
  #0     7      7            7             7        ✅
  #1     2      2            2             2        ✅
  #2     1      1            1             1        ✅
  #3     0      0            0             0        ✅
  #4     4      4            4             4        ✅
  #5     1      1            1             1        ✅
  #6     4      4            4             4        ✅
  #7     9      9            9             9        ✅
